# The Othello edit applied to the **whole observation history**

**One question, two arms.** [`othello_gpt_probing.ipynb`](othello_gpt_probing.ipynb) applies the
Othello-GPT MLP write at **one timestep** — the edit frame — and the generation barely moves
(Edit Index −0.684 → −0.538). Objects move at constant velocity here, so displacing one object by a
constant `δ` at *every past frame* describes a consistent world in which it simply *was* somewhere
else all along. **Does widening the same write from one frame to all of them fix it?**

| arm | what is written | why it is here |
|---|---|---|
| **Unsteered** | nothing | the baseline both arms are read against |
| **Single-frame write** | the MLP write at the **last** timestep | the sibling notebook's method — the thing being compared against |
| **History write** | the **same** MLP write at **every** history position | the method under test |

That is the whole comparison. Everything below is these three arms plus the step size and applied
layer they need.

**Nothing here uses the renderer, ground truth, or an oracle.** The write is the paper's own rule —
gradient descent on the activation, `x' ← x − α ∂L(p_θ(x), B')/∂x` — applied at residual point `L_s`
and every point after it, exactly as in the sibling notebook. The only change is *how many
timesteps* it is applied to. The per-frame targets come from the probe's own read-out of the
model's residual stream; `δ` comes from that same decoded track.

---

### How to read the numbers here (added 2026-08-18, after this notebook misled once)

Two corrections that apply to every figure below, and to the sibling notebook:

1. **Qualitative samples are spread across the teleport-size range, not chosen as the largest.**
   This editor's effect grows with teleport size while the unsteered baseline is flat, so the four
   largest-teleport episodes sit at the **98th percentile** of the Edit Index distribution (+0.07
   against a −0.54 mean). Picking them showed the best case and read as the typical one.
2. **The Edit Index is blind to direction.** It is a ratio of distances, so it reports how *close*
   the output got and says nothing about whether the change it made pointed the right way. A
   directionally-correct edit that is 5% complete scores about the same as a directionally-random
   one. Every table below therefore also carries **direction cosine** (with its shuffled-pair
   chance level and the angle) and **achieved fraction**, from
   `editability_metrics.direction_report`.

## Definitions

**Metrics** are the canonical set from [`../METRICS_AND_EDITORS.md`](../METRICS_AND_EDITORS.md),
computed by `scripts/editability_metrics.py`. The two that matter most here:

| name | formula | units | better | what it can and cannot see |
|---|---|---|---|---|
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_·` = RMSE to each reference world over the **differing rays** (where the two worlds disagree — the union of target and ghost rays); per sample, then averaged | −1…+1 | ↑ | **+1** = the output *is* the edited world, **−1** = the unedited one, **0** = equidistant *including garbage*. **Blind to direction** — see below |
| **direction cosine** ⭐ | cos between the **achieved** change `(edited₀ − unsteered₀)` and the **required** change `(gt_edited − gt_unedited)`, on the differing rays | — | ↑ | did the change point the right way, regardless of size. Reported with its **shuffled-pair chance level** and as an **angle** |
| **achieved fraction** ⭐ | projection of achieved onto required, ÷ required | — | ↑ | how much of the change was made. 1.0 = complete, 0.05 = 5% complete |
| **Target / Ghost / Collateral RMSE** | RMSE to `gt_edited` over the rays the edited object lands on / vacates / the **other** object occupies | obs | ↓ | did the object appear, leave, and was the other one spared |
| **fidelity ratio** | GT-traj RMSE(arm) ÷ GT-traj RMSE(unsteered) | ratio | ↓ | **>1.05 = the arm moved the index by degrading the output** |

⭐ added 2026-08-18. The Edit Index compresses hard near zero — the registry's own calibration says
**+0.2 means "the object barely moves toward the target; not a landed edit"** — and it cannot
distinguish a small-but-correct change from a random one. That is precisely the distinction this
notebook turns out to need.

**World model.** `W16` — **transformer · window 16**, `d_model=256`, 4 layers, carried `state_span`
61 frames, best val 0.02359 (row copied from
[`../transformers/TRANSFORMER_RUNS.md`](../transformers/TRANSFORMER_RUNS.md)). No new models trained.
**Data.** `datasets/4_fixed_refl_inview`, edits split, `ef = 20`, `K = 15`, **N = 256** — the same
episodes as the sibling notebook.

### Terms

| term | meaning |
|---|---|
| **decoded track** | positions the probe reads from the residual stream at every frame `0…ef−1`. The model's own belief — the only position source used |
| **`δ`** | `target − (decoded_track[ef−1] + step)`, `step` = mean first difference of the decoded track over its last 10 frames |
| **per-frame target** | for frame `t`: the edited object at `decoded_track[t] + δ`, the other object held where the probe already reads it |
| **applied layer `L_s`** | the write is applied at this residual point and **every point after it**, alternating write and compute — the paper's Figure 2C schedule |

In [ ]:
# [1] Setup — same model, same episodes, same metrics as the sibling notebook.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pim").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/othello_gpt"))
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import history_edit as he
import pipeline as pl
from editability_metrics import edit_scorecard, fidelity_ratio
from pim.figures import waterfall_grid
from pim.figures.theme import PALETTE, style_ax

RUN = "W16"
N_SEQ = 1500
N_EDIT = 256
N_FIT = 10  # frames of the decoded track used to estimate the per-frame step
BETA = 1.0  # hold-the-rest weight, as in the sibling notebook
N_STEPS = 100  # gradient steps for the activation write
N_STEPS_OBS = 300  # gradient steps for the observation write

bundle = pl.load(RUN)
model, sim = bundle.model, bundle.sim
EF = bundle.edits.edit_frame
N_POINTS = model.cfg.n_layers + 1
POINT_LABELS = [pl.residual_point_label(i) for i in range(N_POINTS)]

probes, stats = pl.probe_table(bundle, n_seq=N_SEQ, hidden=512, epochs=200, seed=0)
R2 = [next(s["r2"] for s in stats if s["target"] == "pos"
           and s["family"] == "MLP (512 hidden)" and s["point"] == i) for i in range(N_POINTS)]
P = {i: probes[("pos", i)] for i in range(N_POINTS)}  # the probe at every residual point
DECODE_POINT = int(np.argmax(R2))

setup = pl.edit_setup(bundle, n_edit=N_EDIT)
print(f"run              : {RUN}, val {bundle.info.val_loss:.5f}, state_span {model.state_span}")
print(f"probe R² by point: " + "  ".join(f"{v:.3f}" for v in R2))
print(f"decode positions from residual point {DECODE_POINT} "
      f"({POINT_LABELS[DECODE_POINT]}), R² {R2[DECODE_POINT]:.3f}")
print(f"episodes         : {setup.n}, edit frame {EF}, K = {pl.K}")
print(f"teleport distance: mean {setup.zones.teleport.mean():.3f} sim units")

In [ ]:
# [2] Decode the history and build δ. This is everything the write uses: the model's own decoded
#     track, and a displacement derived from it. Ground truth appears only as a printed diagnostic.
obs_hist = bundle.edits.obs[: setup.n, :EF]
oe = bundle.edits.edit_object[: setup.n].astype(int)
tgt_pos = bundle.edits.positions[: setup.n, EF, : pl.N_OBJ, :].astype(np.float32)
gt_track = bundle.edits.positions[: setup.n, :EF, : pl.N_OBJ, :].astype(np.float32)
idx = np.arange(setup.n)

track = he.decode_history_positions(model, P[DECODE_POINT], obs_hist, DECODE_POINT)
delta = he.edit_delta(track, tgt_pos, oe, n_fit=N_FIT)

gt_step = gt_track[:, -1] - gt_track[:, -2]
gt_delta = tgt_pos[idx, oe] - (gt_track[idx, -1, oe] + gt_step[idx, oe])
print(f"decoded track vs GT positions (diagnostic only): RMSE {float(np.sqrt(((track - gt_track) ** 2).mean())):.4f} sim units")
print(f"displacement |δ| : mean {np.linalg.norm(delta, axis=-1).mean():.3f} "
      f"(a GT-derived δ would be {np.linalg.norm(gt_delta, axis=-1).mean():.3f}, "
      f"error {float(np.linalg.norm(delta - gt_delta, axis=-1).mean()):.3f})")

In [ ]:
# [3] Step size and applied layer, chosen by READ-OUT CONVERGENCE — the setting at which the probe
#     actually reaches the per-frame targets. Never chosen by Edit Index: large steps raise the
#     index by degrading the frame, which the fidelity column shows directly.
uns_roll = pl.free_rollout(model, setup.state)
uns_card = edit_scorecard(uns_roll, setup.zones, setup.gt_roll)

GRID = [0.02, 0.05, 0.15, 0.4]
sweep = {}
for ls in range(N_POINTS):
    for a in GRID:
        rec = {}
        r = he.activation_history_edit_rollout(model, setup.state, P, track, delta, oe, ls, pl.K,
                                               alpha=a, n_steps=N_STEPS, beta=BETA, record=rec)
        c = edit_scorecard(r, setup.zones, setup.gt_roll)
        sweep[(ls, a)] = dict(readout=rec[ls]["readout_err_after"],
                              readout0=rec[ls]["readout_err_before"],
                              dx=rec[ls]["delta_norm"] / rec[ls]["x_norm"],
                              ei=c["edit_index"], fid=fidelity_ratio(c, uns_card))

BEST_LS, ALPHA = min(sweep, key=lambda k: sweep[k]["readout"])
print(f"unsteered Edit Index: {uns_card['edit_index']:+.3f}   "
      f"read-out error before any write: {sweep[(0, GRID[0])]['readout0']:.2f} sim units\n")
print(f"{'L_s':>4} {'α':>6} {'read-out':>10} {'‖Δx‖/‖x‖':>10} {'Edit Index':>11} {'fidelity':>9}")
for ls in range(N_POINTS):
    for a in GRID:
        s_ = sweep[(ls, a)]
        flag = "  <- lowest read-out" if (ls, a) == (BEST_LS, ALPHA) else ""
        warn = "  ⚠ degrading" if s_["fid"] > 1.05 else ""
        print(f"{ls:>4} {a:>6} {s_['readout']:>10.3f} {s_['dx']:>10.3f} "
              f"{s_['ei']:>+11.3f} {s_['fid']:>9.3f}{flag}{warn}")
print(f"\noperating point: applied layer {BEST_LS}, α = {ALPHA}")

In [ ]:
# [4] The three arms, and their scorecards including the new direction metrics.
from editability_metrics import direction_report  # noqa: E402

_rl, _, _ = pl.run_arms(bundle, setup, probes, "pos", alpha=0.05, n_steps=100, beta=BETA)
ROLLS = {
    "Unsteered": uns_roll,
    "Single-frame write": _rl["from " + pl.residual_point_label(0)],
    "History write (all 20 frames)": he.activation_history_edit_rollout(
        model, setup.state, P, track, delta, oe, BEST_LS, pl.K,
        alpha=ALPHA, n_steps=N_STEPS, beta=BETA),
}
ARMS = list(ROLLS)

CARDS = {k: edit_scorecard(v, setup.zones, setup.gt_roll) for k, v in ROLLS.items()}
for k in CARDS:
    CARDS[k]["fidelity_ratio"] = fidelity_ratio(CARDS[k], CARDS["Unsteered"])
    CARDS[k].update(direction_report(ROLLS[k][:, 0], uns_roll[:, 0], setup.zones))

for k in ARMS:
    c = CARDS[k]
    print(f"  {k:<32} EI {c['edit_index']:+.3f} -> {c['edit_index_by_step'][-1]:+.3f}   "
          f"dir cos {c['direction_cos']:+.3f} ({c['direction_angle_deg']:.0f}°, chance "
          f"{c['direction_cos_shuffled']:+.3f})   achieved {c['achieved_fraction']:.3f}   "
          f"fidelity {c['fidelity_ratio']:.3f}")

In [ ]:
# [5] Fig 1 — waterfall. Samples are spread across the teleport-size range (see the note at the
#     top): one per quartile band, so the panel shows the typical case rather than the best one.
SAMPLES = pl.representative_samples(setup.zones.teleport, k=4)
fig = waterfall_grid(
    {k: ROLLS[k] for k in ARMS},
    setup.ctx,
    setup.gt_roll,
    title=("Fig 1 — the same MLP write at one frame vs at every frame of the history\n"
           f"transformer · window 16 · N = {setup.n} edits · applied layer {BEST_LS}, α = {ALPHA} · "
           "samples spread across the teleport-size range"),
    sample_idx=SAMPLES,
    target_x=setup.tgt_cx,
    ghost_x=setup.ghost_cx,
    metrics={k: CARDS[k]["edit_index"] for k in ARMS},
    metric_label="Edit Index",
    gt_label="GT (sim clean obs)",
)
plt.show()
print(f"samples {SAMPLES}, teleport distances "
      f"{[f'{setup.zones.teleport[i]:.2f}' for i in SAMPLES]} "
      f"(full range {setup.zones.teleport.min():.2f}–{setup.zones.teleport.max():.2f})")

In [ ]:
# [6] Fig 2 — the edit frame on its own, and what the two metrics each see. Panel (a) is the
#     observation curves; panel (b) separates DIRECTION from MAGNITUDE, which is the distinction
#     the Edit Index cannot make.
smp = SAMPLES[2]
m = setup.zones.differing[smp]
rays = np.arange(setup.zones.gt_edited.shape[1])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), facecolor="white")

ax = axes[0]
ax.fill_between(rays, 0, 1, where=m, color="#cfd8e8", alpha=0.75, step="mid", zorder=0)
ax.plot(rays, setup.zones.gt_unedited[smp], color="#555", ls=":", lw=2.0)
ax.plot(rays, setup.zones.gt_edited[smp], color="#111", lw=2.0)
ax.plot(rays, ROLLS["Unsteered"][smp, 0], color=PALETTE[0], lw=1.8)
ax.plot(rays, ROLLS["History write (all 20 frames)"][smp, 0], color=PALETTE[1], lw=1.8)
ax.axvline(setup.tgt_cx[smp], color="#00b050", lw=1.4)
ax.axvline(setup.ghost_cx[smp], color="#d55e00", ls="--", lw=1.4)
ax.set_xlabel("ray index (the 1D scan)")
ax.set_ylabel("observation intensity")
ax.set_ylim(-0.02, 1.02)
ax.set_title(f"(a) the edit frame, sample {smp} "
             f"(teleport {setup.zones.teleport[smp]:.2f}, a median episode)", fontsize=10)
ax.grid(alpha=0.25)
style_ax(ax)
h = [plt.Line2D([], [], color="#555", ls=":", lw=2.0), plt.Line2D([], [], color="#111", lw=2.0),
     plt.Line2D([], [], color=PALETTE[0], lw=1.8), plt.Line2D([], [], color=PALETTE[1], lw=1.8),
     plt.Rectangle((0, 0), 1, 1, color="#cfd8e8"), plt.Line2D([], [], color="#00b050", lw=1.4),
     plt.Line2D([], [], color="#d55e00", ls="--", lw=1.4)]
ax.legend(h, ["GT — unedited world", "GT — edited world (target)", "model, unsteered",
              "model, history write", "rays the Edit Index scores", "target", "ghost"],
          fontsize=7.5, loc="upper right", handlelength=2.2)

ax = axes[1]
xs = np.arange(len(ARMS[1:]))
w = 0.38
cos = [CARDS[k]["direction_cos"] for k in ARMS[1:]]
frac = [CARDS[k]["achieved_fraction"] for k in ARMS[1:]]
ax.bar(xs - w / 2, cos, w, color=PALETTE[0], edgecolor="#333", lw=0.6,
       label="direction cosine — did it point the right way")
ax.bar(xs + w / 2, frac, w, color=PALETTE[3], edgecolor="#333", lw=0.6,
       label="achieved fraction — how much of the change was made")
ax.axhline(CARDS[ARMS[1]]["direction_cos_shuffled"], color="#d55e00", ls=":", lw=1.8,
           label="direction cosine, shuffled-pair chance level")
ax.axhline(1.0, color="#555", ls="--", lw=1.5, label="a complete edit")
ax.set_xticks(xs)
ax.set_xticklabels(ARMS[1:], fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("value (both quantities are dimensionless, 0–1)")
ax.set_title("(b) direction and magnitude, measured separately", fontsize=10)
ax.grid(alpha=0.25, axis="y")
ax.legend(fontsize=7.5, handlelength=2.4, loc="upper center")
style_ax(ax)

fig.suptitle("Fig 2 — why the figure and the Edit Index disagree: the change points the right way "
             "and is a few percent of the way there",
             fontsize=11.5, y=1.04)
fig.tight_layout()
plt.show()

for k in ARMS[1:]:
    c = CARDS[k]
    print(f"  {k:<32} direction cos {c['direction_cos']:+.3f} = {c['direction_angle_deg']:.0f}° "
          f"(chance {c['direction_cos_shuffled']:+.3f})   achieved {c['achieved_fraction']:.1%} "
          f"of the required change")

In [ ]:
# [7] Fig 3 — Edit Index across the rollout, and the scorecard table.
steps = np.arange(pl.K)
COLS = {"Unsteered": ("#555555", ":"), "Single-frame write": (PALETTE[1], "-"),
        "History write (all 20 frames)": (PALETTE[0], "-")}
fig, ax = plt.subplots(figsize=(8.5, 4.8), facecolor="white")
for k, (col, ls_) in COLS.items():
    ax.plot(steps, CARDS[k]["edit_index_by_step"], lw=2.3, marker="o", ms=4, color=col, ls=ls_,
            label=k)
ax.axhline(0.0, color="#999", lw=1)
ax.set_xlabel("rollout step   (step 0 = the edit frame)")
ax.set_ylabel("Edit Index   −1 … +1, ↑ better")
ax.set_ylim(-0.8, 0.2)
ax.grid(alpha=0.25)
ax.legend(fontsize=9, handlelength=2.8)
style_ax(ax)
fig.suptitle("Fig 3 — Edit Index across the rollout\n"
             f"transformer · window 16 · N = {setup.n} edits · applied layer {BEST_LS}, α = {ALPHA}",
             fontsize=11.5, y=1.03)
fig.tight_layout()
plt.show()

rows = ["| arm | Edit Index (step 0) | Edit Index (step 14) | direction cos (angle) | achieved "
        "fraction | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | fidelity |",
        "|---|---|---|---|---|---|---|---|---|"]
for k in ARMS:
    c = CARDS[k]
    dc = "—" if k == "Unsteered" else \
        f"{c['direction_cos']:+.3f} ({c['direction_angle_deg']:.0f}°)"
    af = "—" if k == "Unsteered" else f"{c['achieved_fraction']:.3f}"
    note = " ⚠" if c["fidelity_ratio"] > 1.05 else ""
    rows.append(f"| {k} | {c['edit_index']:+.3f} | {c['edit_index_by_step'][-1]:+.3f} | {dc} "
                f"| {af} | {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} "
                f"| {c['collateral_rmse']:.3f} | {c['fidelity_ratio']:.3f}{note} |")
display(Markdown("**Table 1 — the three arms**\n\n" + "\n".join(rows)
                 + f"\n\nDirection-cosine chance level (shuffled pairs): "
                   f"**{CARDS[ARMS[1]]['direction_cos_shuffled']:+.3f}**. For reference, the "
                   f"unsteered output's own zone errors are the starting point every arm moves "
                   f"from, and a perfect edit would score 0.000 on all three zone RMSEs."))